# Mask R-CNN + ResNet50-FPN — Segmentación de Vértebras

Segmentación multiclase de vértebras en radiografías de columna.  
**Clases:** 23 (background + C3–C7 + T1–T12 + L1–L5)  
**Hardware objetivo:** NVIDIA GTX 1650 4GB — FP16 obligatorio, batch_size=1

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import albumentations as A
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# === CONFIGURACIÓN ===

DATASET_ROOT    = '../Scoliosis_Dataset'
DATASET_INDEX   = os.path.join(DATASET_ROOT, 'indice_dataset.csv')
CHECKPOINTS_DIR = 'checkpoints'
MODELS_DIR      = '../models'

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SIZE  = (512, 1024)   # (ancho, alto)
NUM_CLASSES  = 18
BASE_LR      = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 1
SEED         = 42

CLASS_NAMES = {
    1: 'T1',  2: 'T2',  3: 'T3',  4: 'T4',   5: 'T5',
    6: 'T6',  7: 'T7',  8: 'T8',  9: 'T9',  10: 'T10',
    11: 'T11', 12: 'T12',
    13: 'L1', 14: 'L2', 15: 'L3', 16: 'L4', 17: 'L5',
}

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Dispositivo: {DEVICE}')

---
## Sección 1 — Preprocesamiento

### Funciones de carga

In [ ]:
def load_image(image_path: str) -> np.ndarray:
    """Carga imagen RGB uint8 desde disco."""
    return np.array(Image.open(image_path).convert('RGB'))


def load_mask(mask_path: str) -> np.ndarray:
    """Carga máscara 16-bit uint16 sin truncar IDs de clase."""
    return cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)


def load_binary_mask(binary_mask_path: str) -> np.ndarray:
    """Carga máscara binaria y la binariza para neutralizar artefactos JPEG."""
    raw = cv2.imread(binary_mask_path, cv2.IMREAD_GRAYSCALE)
    return (raw > 127).astype(np.uint8)


def load_dataset_index(csv_path: str) -> pd.DataFrame:
    """Carga dataset_index.csv como fuente canónica de rutas."""
    return pd.read_csv(csv_path)

### Transformaciones

In [ ]:
def to_grayscale(image: np.ndarray) -> np.ndarray:
    """Conversión perceptual: L = 0.299R + 0.587G + 0.114B."""
    return (0.299 * image[:, :, 0]
            + 0.587 * image[:, :, 1]
            + 0.114 * image[:, :, 2]).astype(np.uint8)


def map_entity_ids(mask: np.ndarray) -> np.ndarray:
    """Mapea IDs 23–35 (Entity X) a 0 (background). Devuelve uint8."""
    result = mask.copy()
    result[result > 22] = 0
    return result.astype(np.uint8)


def compute_roi(binary_mask: np.ndarray, margin: float = 0.10) -> tuple:
    """Bounding box de la columna sobre la máscara binaria con margen ≥10%."""
    rows = np.any(binary_mask, axis=1)
    cols = np.any(binary_mask, axis=0)
    if not rows.any():
        h, w = binary_mask.shape
        return (0, 0, w, h)
    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    h, w = binary_mask.shape
    dy = max(1, int((y2 - y1) * margin))
    dx = max(1, int((x2 - x1) * margin))
    return (
        max(0, x1 - dx),
        max(0, y1 - dy),
        min(w, x2 + dx),
        min(h, y2 + dy),
    )


def crop_to_roi(image: np.ndarray, mask: np.ndarray, roi: tuple) -> tuple:
    """Aplica el mismo crop a imagen (H,W) y máscara (H,W)."""
    x1, y1, x2, y2 = roi
    img_crop  = image[y1:y2, x1:x2]
    mask_crop = mask[y1:y2, x1:x2]
    return img_crop, mask_crop


def resize_pair(image: np.ndarray, mask: np.ndarray, target_size: tuple) -> tuple:
    """Resize: bilinear para imagen, nearest neighbor para máscara."""
    w, h = target_size
    img_r  = cv2.resize(image, (w, h), interpolation=cv2.INTER_LINEAR)
    mask_r = cv2.resize(mask,  (w, h), interpolation=cv2.INTER_NEAREST)
    return img_r, mask_r


def apply_clahe(image: np.ndarray,
                clip_limit: float = 2.0,
                tile_grid: tuple = (8, 8)) -> np.ndarray:
    """CLAHE sobre imagen monocanal uint8."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(image)


def replicate_to_3ch(image: np.ndarray) -> np.ndarray:
    """Replica canal gris (H,W) a 3 canales (H,W,3) para compatibilidad ImageNet."""
    return np.stack([image, image, image], axis=-1)


def normalize_image(image: np.ndarray) -> torch.Tensor:
    """Estandarización con stats ImageNet. Entrada uint8 (H,W,3) → Tensor float32 (3,H,W)."""
    img = image.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img  = (img - mean) / std
    return torch.from_numpy(img.transpose(2, 0, 1))


def semantic_to_instance(mask: np.ndarray) -> dict:
    """
    Convierte máscara semántica (ID por píxel) a formato instancia para Mask R-CNN.
    Cada clase = una instancia única por imagen.
    Retorna dict con boxes (N,4), masks (N,H,W), labels (N,).
    """
    class_ids = np.unique(mask)
    class_ids = class_ids[class_ids > 0]
    boxes, masks, labels = [], [], []

    for cid in class_ids:
        binary = (mask == cid).astype(np.uint8)
        ys, xs = np.where(binary)
        if len(xs) == 0:
            continue
        x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
        if x2 <= x1 or y2 <= y1:
            continue
        boxes.append([x1, y1, x2, y2])
        masks.append(binary)
        labels.append(int(cid))

    if not boxes:
        h, w = mask.shape
        return {
            'boxes':  torch.zeros((0, 4), dtype=torch.float32),
            'masks':  torch.zeros((0, h, w), dtype=torch.uint8),
            'labels': torch.zeros(0, dtype=torch.int64),
        }

    return {
        'boxes':  torch.tensor(boxes, dtype=torch.float32),
        'masks':  torch.tensor(np.stack(masks), dtype=torch.uint8),
        'labels': torch.tensor(labels, dtype=torch.int64),
    }

### Augmentation

In [ ]:
def build_augmentation_pipeline() -> A.Compose:
    """
    Pipeline de augmentation sincronizada imagen-máscara.
    Rotación ±15° vía Affine (reemplaza ShiftScaleRotate deprecado),
    distorsión de grilla, dropout grueso para simular artefactos de radiografía
    y ruido gaussiano con API actual de albumentations.
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(
            rotate=(-15, 15),
            scale=(0.85, 1.15),
            mode=cv2.BORDER_CONSTANT,
            cval=0,
            cval_mask=0,
            p=0.6,
        ),
        A.ElasticTransform(alpha=1, sigma=50, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.RandomBrightnessContrast(p=0.4),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        A.CoarseDropout(
            num_holes_range=(1, 4),
            hole_height_range=(20, 60),
            hole_width_range=(20, 60),
            fill=0,
            p=0.2,
        ),
    ], additional_targets={'mask': 'mask'})


def apply_augmentation(image: np.ndarray,
                       mask: np.ndarray,
                       pipeline: A.Compose) -> tuple:
    """Augmentation sincronizada imagen-máscara."""
    result = pipeline(image=image, mask=mask)
    return result['image'], result['mask']

### Split y Dataset

In [ ]:
def split_dataset(df: pd.DataFrame,
                  train: float = 0.70,
                  val: float = 0.15,
                  seed: int = 42) -> tuple:
    """Split 70/15/15 estratificado por columna 'grupo' (Normal/Scoliosis)."""
    train_df, temp_df = train_test_split(
        df, train_size=train, stratify=df['grupo'], random_state=seed
    )
    val_ratio = val / (1.0 - train)
    val_df, test_df = train_test_split(
        temp_df, train_size=val_ratio, stratify=temp_df['grupo'], random_state=seed
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


class SpineDataset(Dataset):
    """
    Dataset de vértebras. Aplica el pipeline de preprocesamiento completo.
    mode='train' aplica augmentation; 'val'/'test' no.
    Retorna (image_tensor, target) donde target incluye 'full_mask' para métricas.
    """

    def __init__(self, df: pd.DataFrame, mode: str,
                 aug_pipeline: A.Compose = None,
                 target_size: tuple = (512, 1024),
                 dataset_root: str = DATASET_ROOT):
        self.df           = df
        self.mode         = mode
        self.aug_pipeline = aug_pipeline
        self.target_size  = target_size
        self.root         = dataset_root

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple:
        row = self.df.iloc[idx]

        # Carga
        image       = load_image(os.path.join(self.root, row['ruta_radiografia']))
        mask        = load_mask(os.path.join(self.root, row['ruta_mascara_multiclase_id_png']))
        binary_mask = load_binary_mask(os.path.join(self.root, row['ruta_mascara_binaria']))

        # Paso 1: escala de grises
        image = to_grayscale(image)

        # Paso 2: mapeo de IDs fuera de rango
        mask = map_entity_ids(mask)

        # Paso 3: ROI crop (a resolución original)
        roi           = compute_roi(binary_mask)
        image, mask   = crop_to_roi(image, mask, roi)

        # Paso 4: resize
        image, mask   = resize_pair(image, mask, self.target_size)

        # Paso 5: CLAHE
        image = apply_clahe(image)

        # Paso 6: augmentation (solo train)
        if self.mode == 'train' and self.aug_pipeline is not None:
            image, mask = apply_augmentation(image, mask, self.aug_pipeline)

        # Paso 7: normalización
        image_tensor = normalize_image(replicate_to_3ch(image))

        # Target para Mask R-CNN
        target = semantic_to_instance(mask)
        # full_mask se usa solo en métricas, no se pasa al modelo
        target['full_mask'] = torch.from_numpy(mask.copy())
        target['image_id']  = torch.tensor([idx])

        return image_tensor, target


def collate_fn(batch: list) -> tuple:
    """Mask R-CNN espera lista de tensores, no batch tensorial apilado."""
    images  = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    return images, targets


def create_dataloaders(train_ds: Dataset,
                       val_ds: Dataset,
                       test_ds: Dataset,
                       batch_size: int = 1) -> tuple:
    """Crea los 3 DataLoaders con collate_fn para Mask R-CNN."""
    train_dl = DataLoader(train_ds, batch_size=batch_size,
                          shuffle=True,  collate_fn=collate_fn, num_workers=2)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size,
                          shuffle=False, collate_fn=collate_fn, num_workers=2)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size,
                          shuffle=False, collate_fn=collate_fn, num_workers=2)
    return train_dl, val_dl, test_dl

---
## Sección 2 — Procesamiento (Entrenamiento)

### Construcción del modelo

In [ ]:
def build_model(num_classes: int = NUM_CLASSES) -> torchvision.models.detection.MaskRCNN:
    """
    Carga Mask R-CNN + ResNet50-FPN preentrenado en COCO.
    Reemplaza box_predictor y mask_predictor con num_classes.
    """
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1)

    # Reemplazar la cabeza de clasificación de bounding box
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)

    # Reemplazar la cabeza de segmentación de máscara
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer     = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model

### Control del encoder

In [ ]:
def freeze_encoder(model) -> None:
    """Congela todos los parámetros del backbone ResNet50."""
    for param in model.backbone.body.parameters():
        param.requires_grad = False


def unfreeze_block(model, block_idx: int) -> None:
    """
    Descongela layer{block_idx} del backbone ResNet50.
    block_idx en [1, 2, 3, 4]. Llamar en orden descendente: 4 → 3 → 2 → 1.
    """
    layer = getattr(model.backbone.body, f'layer{block_idx}')
    for param in layer.parameters():
        param.requires_grad = True


def get_param_groups(model, base_lr: float = BASE_LR) -> list:
    """
    Construye grupos de parámetros con LR diferencial por bloque del encoder.
    Solo incluye parámetros con requires_grad=True.
    Decoder/FPN/RPN/cabezas: base_lr
    layer4: base_lr × 0.1
    layer3: base_lr × 0.01
    layer2: base_lr × 0.001
    layer1: base_lr × 0.0001
    """
    lr_map = {
        'backbone.body.layer1': base_lr * 0.0001,
        'backbone.body.layer2': base_lr * 0.001,
        'backbone.body.layer3': base_lr * 0.01,
        'backbone.body.layer4': base_lr * 0.1,
    }

    layer_params   = {k: [] for k in lr_map}
    decoder_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        matched = False
        for layer_name in lr_map:
            if name.startswith(layer_name):
                layer_params[layer_name].append(param)
                matched = True
                break
        if not matched:
            decoder_params.append(param)

    groups = []
    if decoder_params:
        groups.append({'params': decoder_params, 'lr': base_lr})
    for layer_name, params in layer_params.items():
        if params:
            groups.append({'params': params, 'lr': lr_map[layer_name]})

    return groups

### Optimizador y scheduling

In [ ]:
def build_optimizer(param_groups: list,
                    weight_decay: float = WEIGHT_DECAY) -> torch.optim.Optimizer:
    """SGD con momentum=0.9 y weight_decay sobre los param groups diferenciales."""
    return torch.optim.SGD(param_groups, momentum=0.9, weight_decay=weight_decay)


def build_scheduler(optimizer: torch.optim.Optimizer,
                    patience: int = 3,
                    factor: float = 0.5) -> torch.optim.lr_scheduler.ReduceLROnPlateau:
    """Reduce el LR a la mitad si val_loss no mejora en patience epochs."""
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=factor, patience=patience, verbose=True
    )

### Loops de entrenamiento

In [ ]:
def train_one_epoch(model, dataloader, optimizer, scaler, device) -> float:
    """
    Loop de entrenamiento con mixed precision (FP16).
    Mask R-CNN en modo train devuelve un dict de losses.
    Retorna el loss promedio del epoch.
    """
    model.train()
    total_loss = 0.0

    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        model_targets = [
            {k: v.to(device) for k, v in t.items()
             if k not in ('full_mask', 'image_id')}
            for t in targets
        ]

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss_dict = model(images, model_targets)
            losses    = sum(loss_dict.values())

        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += losses.item()

    return total_loss / len(dataloader)


def validate_one_epoch(model, dataloader, device) -> float:
    """
    Loop de validación. Mask R-CNN necesita estar en modo train para devolver losses;
    se desactivan los gradientes con torch.no_grad().
    Retorna el loss promedio del epoch.
    """
    model.train()
    total_loss = 0.0

    with torch.no_grad():
        for images, targets in dataloader:
            images = [img.to(device) for img in images]
            model_targets = [
                {k: v.to(device) for k, v in t.items()
                 if k not in ('full_mask', 'image_id')}
                for t in targets
            ]
            loss_dict  = model(images, model_targets)
            total_loss += sum(loss_dict.values()).item()

    return total_loss / len(dataloader)

### Checkpointing

In [ ]:
def save_checkpoint(model, path: str, val_loss: float) -> None:
    """Guarda pesos completos del modelo cuando val_loss mejora."""
    torch.save(model.state_dict(), path)
    print(f'  Checkpoint guardado: {path}  (val_loss={val_loss:.4f})')


def load_checkpoint(model, path: str):
    """Carga el mejor checkpoint de una fase."""
    model.load_state_dict(torch.load(path, map_location='cpu'))
    return model


def save_final_model(model, path: str) -> None:
    """Guarda el modelo final entrenado en un .pth fijo. Se llama una vez al terminar todas las fases."""
    torch.save(model.state_dict(), path)
    print(f'Modelo final guardado en: {path}')

### Entrenamiento por fases

In [ ]:
def train_phase(model, train_dl, val_dl, optimizer, scheduler, device,
                max_epochs: int, patience: int = 7,
                checkpoint_path: str = 'checkpoint.pth') -> float:
    """
    Entrena una fase completa con:
      - ReduceLROnPlateau (stepping por val_loss)
      - Early stopping (patience epochs sin mejora en val_loss)
      - Checkpointing (guarda cuando val_loss mejora)
    Carga los mejores pesos al finalizar la fase.
    Retorna el mejor val_loss de la fase.
    """
    scaler          = torch.amp.GradScaler('cuda')
    best_val_loss   = float('inf')
    epochs_no_impr  = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_dl, optimizer, scaler, device)
        val_loss   = validate_one_epoch(model, val_dl, device)
        scheduler.step(val_loss)

        print(f'  Epoch {epoch}/{max_epochs} — '
              f'train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            epochs_no_impr = 0
            save_checkpoint(model, checkpoint_path, val_loss)
        else:
            epochs_no_impr += 1
            if epochs_no_impr >= patience:
                print(f'  Early stopping en epoch {epoch}')
                break

    model = load_checkpoint(model, checkpoint_path)
    return best_val_loss


def run_progressive_training(model, train_dl, val_dl, device,
                             base_lr: float = BASE_LR,
                             weight_decay: float = WEIGHT_DECAY):
    """
    Orquesta 4 fases de descongelamiento progresivo:
      Fase 1 — Encoder congelado       (30 epochs max)
      Fase 2 — Descongelar layer4      (20 epochs max)
      Fase 3 — Descongelar layer3      (15 epochs max)
      Fase 4 — Descongelar layer2      (10 epochs max)
    Retorna el modelo con los mejores pesos de la última fase.
    """
    phases = [
        {'name': 'Fase 1 — Encoder congelado',  'unfreeze': None, 'max_epochs': 30},
        {'name': 'Fase 2 — Descongelar layer4', 'unfreeze': 4,    'max_epochs': 20},
        {'name': 'Fase 3 — Descongelar layer3', 'unfreeze': 3,    'max_epochs': 15},
        {'name': 'Fase 4 — Descongelar layer2', 'unfreeze': 2,    'max_epochs': 10},
    ]

    freeze_encoder(model)

    for i, phase in enumerate(phases, start=1):
        print(f'\n=== {phase["name"]} ===')

        if phase['unfreeze'] is not None:
            unfreeze_block(model, phase['unfreeze'])

        param_groups = get_param_groups(model, base_lr)
        optimizer    = build_optimizer(param_groups, weight_decay)
        scheduler    = build_scheduler(optimizer)
        ckpt_path    = os.path.join(CHECKPOINTS_DIR, f'phase{i}_best.pth')

        best = train_phase(model, train_dl, val_dl, optimizer, scheduler,
                           device, phase['max_epochs'], checkpoint_path=ckpt_path)
        print(f'  Mejor val_loss fase {i}: {best:.4f}')

    return model

---
## Sección 3 — Métricas

### Métricas de pixel (Dice e IoU)

In [ ]:
def compute_dice(pred: np.ndarray, gt: np.ndarray) -> float:
    """Dice entre dos máscaras binarias de la misma clase."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    intersection = (pred & gt).sum()
    denom = pred.sum() + gt.sum()
    return 2.0 * intersection / denom if denom > 0 else 1.0


def compute_iou(pred: np.ndarray, gt: np.ndarray) -> float:
    """IoU entre dos máscaras binarias de la misma clase."""
    pred, gt = pred.astype(bool), gt.astype(bool)
    intersection = (pred & gt).sum()
    union        = (pred | gt).sum()
    return intersection / union if union > 0 else 1.0


def compute_dice_per_class(predictions: list, targets: list,
                           num_classes: int = NUM_CLASSES) -> dict:
    """
    Dice por clase sobre el test set completo.
    Excluye clases ausentes en cada imagen (columna parcial).
    Para clases con múltiples predicciones toma la de mayor score.
    """
    scores = {c: [] for c in range(1, num_classes)}

    for pred, target in zip(predictions, targets):
        full_mask    = target['full_mask'].numpy()
        present      = np.unique(full_mask)
        present      = present[present > 0]
        pred_labels  = pred['labels'].numpy()
        pred_masks   = pred['masks'].numpy()   # (N, 1, H, W)
        pred_scores  = pred['scores'].numpy()

        for c in present:
            gt_binary = (full_mask == c).astype(np.uint8)
            idx_c     = np.where(pred_labels == c)[0]
            if len(idx_c) == 0:
                pred_binary = np.zeros_like(gt_binary)
            else:
                best        = idx_c[np.argmax(pred_scores[idx_c])]
                pred_binary = (pred_masks[best, 0] > 0.5).astype(np.uint8)
            scores[c].append(compute_dice(pred_binary, gt_binary))

    return {c: float(np.mean(v)) for c, v in scores.items() if v}


def compute_mean_dice(dice_per_class: dict) -> float:
    """Promedio de Dice sobre todas las clases presentes en el test set."""
    return float(np.mean(list(dice_per_class.values()))) if dice_per_class else 0.0


def compute_iou_per_class(predictions: list, targets: list,
                          num_classes: int = NUM_CLASSES) -> dict:
    """IoU por clase sobre el test set. Misma lógica que compute_dice_per_class."""
    scores = {c: [] for c in range(1, num_classes)}

    for pred, target in zip(predictions, targets):
        full_mask   = target['full_mask'].numpy()
        present     = np.unique(full_mask)
        present     = present[present > 0]
        pred_labels = pred['labels'].numpy()
        pred_masks  = pred['masks'].numpy()
        pred_scores = pred['scores'].numpy()

        for c in present:
            gt_binary = (full_mask == c).astype(np.uint8)
            idx_c     = np.where(pred_labels == c)[0]
            if len(idx_c) == 0:
                pred_binary = np.zeros_like(gt_binary)
            else:
                best        = idx_c[np.argmax(pred_scores[idx_c])]
                pred_binary = (pred_masks[best, 0] > 0.5).astype(np.uint8)
            scores[c].append(compute_iou(pred_binary, gt_binary))

    return {c: float(np.mean(v)) for c, v in scores.items() if v}


def compute_miou(predictions: list, targets: list,
                 num_classes: int = NUM_CLASSES) -> float:
    """mIoU global sobre clases presentes en el test set."""
    iou_per_class = compute_iou_per_class(predictions, targets, num_classes)
    return float(np.mean(list(iou_per_class.values()))) if iou_per_class else 0.0

### Métricas de detección (AP)

In [ ]:
def _compute_ap(recalls: np.ndarray, precisions: np.ndarray) -> float:
    """Area bajo la curva precision-recall (interpolación continua)."""
    recalls    = np.concatenate(([0.0], recalls,    [1.0]))
    precisions = np.concatenate(([1.0], precisions, [0.0]))
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])
    idx = np.where(recalls[1:] != recalls[:-1])[0] + 1
    return float(np.sum((recalls[idx] - recalls[idx - 1]) * precisions[idx]))


def compute_ap50_per_class(predictions: list, targets: list,
                           num_classes: int = NUM_CLASSES,
                           iou_threshold: float = 0.5) -> dict:
    """
    AP@50 por clase.
    Detección correcta (TP) si IoU entre predicción y GT ≥ iou_threshold.
    Cada GT puede ser emparejado como máximo una vez (el más solapado).
    Excluye clases ausentes en todo el test set.
    """
    ap_per_class = {}

    for c in range(1, num_classes):
        all_entries = []   # (score, tp_flag)
        n_gt        = 0

        for pred, target in zip(predictions, targets):
            full_mask = target['full_mask'].numpy()
            gt_binary = (full_mask == c).astype(bool)
            has_gt    = gt_binary.any()
            if has_gt:
                n_gt += 1

            pred_labels = pred['labels'].numpy()
            pred_masks  = pred['masks'].numpy()
            pred_scores = pred['scores'].numpy()
            idx_c       = np.where(pred_labels == c)[0]
            if len(idx_c) == 0:
                continue

            # Ordenar por score descendente
            idx_c_sorted = sorted(idx_c, key=lambda x: -pred_scores[x])
            gt_matched   = False

            for i in idx_c_sorted:
                score = pred_scores[i]
                pm    = (pred_masks[i, 0] > 0.5).astype(bool)
                if has_gt and not gt_matched:
                    inter = (pm & gt_binary).sum()
                    union = (pm | gt_binary).sum()
                    iou   = inter / union if union > 0 else 0.0
                    if iou >= iou_threshold:
                        all_entries.append((score, 1))
                        gt_matched = True
                        continue
                all_entries.append((score, 0))

        if n_gt == 0:
            continue
        if not all_entries:
            ap_per_class[c] = 0.0
            continue

        all_entries.sort(key=lambda x: -x[0])
        tp_arr    = np.array([e[1] for e in all_entries])
        tp_cum    = np.cumsum(tp_arr)
        fp_cum    = np.cumsum(1 - tp_arr)
        recalls   = tp_cum / n_gt
        precs     = tp_cum / (tp_cum + fp_cum)
        ap_per_class[c] = _compute_ap(recalls, precs)

    return ap_per_class


def compute_map(ap_per_class: dict) -> float:
    """mAP global: promedio de AP@50 sobre todas las clases evaluadas."""
    return float(np.mean(list(ap_per_class.values()))) if ap_per_class else 0.0

### Evaluación completa

In [ ]:
def run_inference(model, dataloader, device) -> tuple:
    """
    Inferencia sobre el test set en modo eval.
    Retorna (predictions, targets) como listas de dicts en CPU.
    predictions[i]: {boxes, labels, scores, masks}
    targets[i]:     {full_mask, labels, boxes, masks}
    """
    model.eval()
    all_predictions, all_targets = [], []

    with torch.no_grad():
        for images, targets in dataloader:
            images  = [img.to(device) for img in images]
            outputs = model(images)
            for output, target in zip(outputs, targets):
                all_predictions.append({k: v.cpu() for k, v in output.items()})
                all_targets.append({k: v.cpu() for k, v in target.items()})

    return all_predictions, all_targets


def evaluate_model(predictions: list, targets: list,
                   num_classes: int = NUM_CLASSES) -> dict:
    """Calcula todas las métricas obligatorias sobre el test set."""
    dice_per_class = compute_dice_per_class(predictions, targets, num_classes)
    iou_per_class  = compute_iou_per_class(predictions, targets, num_classes)
    mean_dice      = compute_mean_dice(dice_per_class)
    miou           = compute_miou(predictions, targets, num_classes)
    ap50_per_class = compute_ap50_per_class(predictions, targets, num_classes)
    map_score      = compute_map(ap50_per_class)

    return {
        'dice_per_class':  dice_per_class,
        'iou_per_class':   iou_per_class,
        'mean_dice':       mean_dice,
        'miou':            miou,
        'ap50_per_class':  ap50_per_class,
        'map':             map_score,
    }


def print_metrics_report(metrics: dict) -> None:
    """Imprime tabla resumen de métricas por clase y globales."""
    header = f"{'Clase':<8} {'Dice':>8} {'IoU':>8} {'AP@50':>8}"
    print(header)
    print('-' * len(header))

    for c in range(1, NUM_CLASSES):
        name = CLASS_NAMES.get(c, f'ID{c}')
        dice = metrics['dice_per_class'].get(c, float('nan'))
        iou  = metrics['iou_per_class'].get(c, float('nan'))
        ap50 = metrics['ap50_per_class'].get(c, float('nan'))
        print(f"{name:<8} {dice:>8.4f} {iou:>8.4f} {ap50:>8.4f}")

    print('-' * len(header))
    print(f"{'mean':<8} {metrics['mean_dice']:>8.4f} {metrics['miou']:>8.4f} {metrics['map']:>8.4f}")
    print()
    print(f"mean Dice: {metrics['mean_dice']:.4f}")
    print(f"mIoU:      {metrics['miou']:.4f}")
    print(f"mAP@50:    {metrics['map']:.4f}")

### Visualización de predicciones

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def visualize_predictions(predictions, targets, test_ds, n=4, score_thr=0.5, seed=42):
    """
    Muestra n ejemplos aleatorios del test set con overlay de segmentacion predicha.

    predictions : list[dict]  – salida de run_inference (masks, labels, scores)
    targets     : list[dict]  – ground truth correspondiente
    test_ds     : SpineDataset del test set (accede a test_ds.df para paths)
    n           : numero de ejemplos a mostrar
    score_thr   : umbral minimo de confianza para dibujar una mascara
    """
    random.seed(seed)
    indices = random.sample(range(len(predictions)), min(n, len(predictions)))

    # Paleta fija por clase (background=0 queda negro/transparente)
    rng = np.random.RandomState(0)
    palette = rng.randint(80, 230, size=(NUM_CLASSES, 3), dtype=np.uint8)
    palette[0] = [0, 0, 0]

    fig, axes = plt.subplots(1, len(indices), figsize=(5 * len(indices), 12))
    if len(indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        # Imagen original en RGB
        img_path = test_ds.df.iloc[idx]["ruta_radiografia"]
        img = np.array(Image.open(img_path).convert("RGB"))
        overlay = img.copy().astype(np.float32)

        pred   = predictions[idx]
        masks  = pred["masks"]   # ndarray float32 (N, H, W) valores [0,1]
        labels = pred["labels"]  # ndarray int     (N,)
        scores = pred["scores"]  # ndarray float   (N,)

        legend_patches = []
        seen_labels    = set()

        for m, lab, sc in zip(masks, labels, scores):
            if sc < score_thr or lab == 0 or int(lab) >= NUM_CLASSES:
                continue
            binary = m > 0.5
            color  = palette[int(lab)].astype(np.float32)
            overlay[binary] = overlay[binary] * 0.45 + color * 0.55
            if lab not in seen_labels:
                name  = CLASS_NAMES.get(int(lab), f"cls_{lab}")
                patch = mpatches.Patch(
                    color=color / 255.0,
                    label=f"{name}  {sc:.2f}"
                )
                legend_patches.append(patch)
                seen_labels.add(lab)

        ax.imshow(overlay.astype(np.uint8))
        ax.set_title(f"Test sample #{idx}", fontsize=9)
        ax.axis("off")
        if legend_patches:
            ax.legend(
                handles=legend_patches,
                loc="lower right",
                fontsize=6,
                framealpha=0.75,
                ncol=2,
            )

    fig.suptitle("Predicciones — Test Set", fontsize=13)
    plt.tight_layout()
    plt.show()


---
## Pipeline

In [ ]:
# === PREPROCESAMIENTO ===
index_df                      = load_dataset_index(DATASET_INDEX)
train_df, val_df, test_df     = split_dataset(index_df, seed=SEED)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

aug_pipeline = build_augmentation_pipeline()
train_ds     = SpineDataset(train_df, mode='train', aug_pipeline=aug_pipeline)
val_ds       = SpineDataset(val_df,   mode='val')
test_ds      = SpineDataset(test_df,  mode='test')

train_dl, val_dl, test_dl = create_dataloaders(train_ds, val_ds, test_ds, BATCH_SIZE)

# === PROCESAMIENTO (ENTRENAMIENTO) ===
model = build_model(NUM_CLASSES).to(DEVICE)
model = run_progressive_training(model, train_dl, val_dl, DEVICE)

# === GUARDAR MODELO FINAL ===
save_final_model(model, os.path.join(MODELS_DIR, 'maskrcnn_resnet50_fpn_best.pth'))

# === MÉTRICAS ===
predictions, targets = run_inference(model, test_dl, DEVICE)
metrics              = evaluate_model(predictions, targets)
print_metrics_report(metrics)
visualize_predictions(predictions, targets, test_ds)